<a href="https://colab.research.google.com/github/sebr22/sebr/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebr22/sebr/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will use a Decision Tree Classifier because my lane is a ranking problem: “Which pages should a content team review first?” The tree can learn how combinations of signals such as impressions, CTR, search position and engagement relate to the review-priority proxy, and its predicted probabilities can be used to rank pages from highest to lowest priority. I chose a Decision Tree because it is relatively simple and interpretable, allowing me to see which features the model uses to make its decisions and compare it fairly against my transparent rule-based baseline. I will start with a shallow tree to avoid unnecessary complexity and only increase the complexity if it provides a meaningful improvement in Precision@K.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I used a grouped-by-client split, with all observations from a given client kept entirely in either the training or test set. This gives 300,880 training rows from 44 clients and 30,557 test rows from 11 clients. I chose this split because the model is intended to rank pages for clients it has not seen during training, so allowing pages from the same client into both sets could make performance look artificially strong.
The split is not time-aware because this analysis deliberately uses March 2026 only: the features and target are defined entirely from information available within that month. Therefore, there are no later months being used to predict March outcomes. The test set acts as a holdout of unseen clients rather than a future time period, providing a stricter test of whether the learned ranking generalises across clients.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET hf_secret (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
SELECT COUNT(*)
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
""")

march_df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position,

        SUM(ga4_sessions) AS ga4_sessions,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions,
        SUM(ga4_total_engagement_sec) AS ga4_total_engagement_sec,

        BOOL_OR(gsc_data_available) AS gsc_data_available,
        BOOL_OR(ga4_data_available) AS ga4_data_available

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )

    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

import numpy as np

march_df["gsc_ctr"] = (
    march_df["gsc_clicks"] /
    march_df["gsc_impressions"].replace(0, np.nan)
).fillna(0)

march_df["engagement_rate"] = (
    march_df["ga4_engaged_sessions"] /
    march_df["ga4_sessions"].replace(0, np.nan)
).fillna(0)

features = [
    "gsc_impressions",
    "gsc_ctr",
    "gsc_avg_position",
    "ga4_sessions",
    "engagement_rate"
]

X = march_df[features].copy()

# Replace infinite values
X = X.replace([np.inf, -np.inf], np.nan)

# Fill missing values
X = X.fillna(0)

print(X.head())
print(X.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   gsc_impressions   gsc_ctr  gsc_avg_position  ga4_sessions  engagement_rate
0            181.0  0.000000          5.147402           0.0              0.0
1             46.0  0.021739          4.828125           0.0              0.0
2            899.0  0.001112          5.145765           0.0              0.0
3             34.0  0.000000          4.909314           0.0              0.0
4           3108.0  0.000000          6.969536           0.0              0.0
(331437, 5)


In [4]:
# Start with a copy so the original feature data is unchanged
baseline_df = march_df.copy()

# Reason-code conditions
baseline_df["high_visibility"] = (
    baseline_df["gsc_impressions"] >= 500
)

baseline_df["low_ctr"] = (
    baseline_df["gsc_ctr"] < 0.01
)

baseline_df["poor_position"] = (
    baseline_df["gsc_avg_position"] > 20
)

baseline_df["no_gsc_data"] = (
    baseline_df["gsc_data_available"] == False
)

# Transparent baseline score
# Higher score = higher priority for review
baseline_df["baseline_score"] = (
    baseline_df["high_visibility"].astype(int) *
    baseline_df["low_ctr"].astype(int) *
    baseline_df["gsc_impressions"]
)

# Reason codes
def get_reason(row):
    if row["no_gsc_data"]:
        return "no_gsc_data"
    elif row["high_visibility"] and row["low_ctr"] and row["poor_position"]:
        return "high_visibility_poor_position"
    elif row["high_visibility"] and row["low_ctr"]:
        return "high_visibility_low_ctr"
    elif row["poor_position"]:
        return "poor_search_position"
    else:
        return "low_priority"

baseline_df["reason_code"] = baseline_df.apply(get_reason, axis=1)

# Rank highest-scoring pages first
baseline_df = baseline_df.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

baseline_df["rank"] = baseline_df.index + 1

# Keep the useful fields in the output
output_cols = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "baseline_score",
    "reason_code",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_ctr",
    "gsc_avg_position"
]

baseline_output = baseline_df[output_cols]

# Save the ranked queue
import os

os.makedirs("work/outputs", exist_ok=True)

baseline_output.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)


In [5]:
baseline_df["baseline_score"]
proxy = baseline_df["baseline_score"].copy()
assert len(X) == len(proxy)


In [6]:
# 1. Prepare March dataset

proxy_df = march_df.copy()

print("Rows:", len(proxy_df))

feature_cols = [
    "gsc_impressions",
    "gsc_ctr",
    "gsc_avg_position",
    "ga4_sessions",
    "engagement_rate"
]

print("Features:", feature_cols)

Rows: 331437
Features: ['gsc_impressions', 'gsc_ctr', 'gsc_avg_position', 'ga4_sessions', 'engagement_rate']


In [7]:
# 2. Engineer model features

import numpy as np

proxy_df["log_impressions"] = np.log1p(
    proxy_df["gsc_impressions"]
)

proxy_df["log_sessions"] = np.log1p(
    proxy_df["ga4_sessions"]
)

proxy_df["log_engagement"] = np.log1p(
    proxy_df["engagement_rate"]
)

model_features = [
    "log_impressions",
    "gsc_ctr",
    "gsc_avg_position",
    "log_sessions",
    "log_engagement"
]

print("Model features:", model_features)

Model features: ['log_impressions', 'gsc_ctr', 'gsc_avg_position', 'log_sessions', 'log_engagement']


In [8]:
# 3. Create March-only proxy target

proxy_df["impression_percentile"] = (
    proxy_df.groupby("client_hash_id")["gsc_impressions"]
    .rank(pct=True)
)

proxy_df["ctr_percentile"] = (
    proxy_df.groupby("client_hash_id")["gsc_ctr"]
    .rank(pct=True)
)

proxy_df["position_percentile"] = (
    proxy_df.groupby("client_hash_id")["gsc_avg_position"]
    .rank(pct=True)
)

proxy_df["engagement_percentile"] = (
    proxy_df.groupby("client_hash_id")["engagement_rate"]
    .rank(pct=True)
)

# A page is "review-worthy" if:
# - it has relatively high visibility within its client
# - AND it has at least one sign of underperformance

proxy_df["target"] = (
    (proxy_df["impression_percentile"] >= 0.75)
    &
    (
        (proxy_df["ctr_percentile"] <= 0.25)
        |
        (proxy_df["position_percentile"] >= 0.75)
        |
        (proxy_df["engagement_percentile"] <= 0.25)
    )
).astype(int)

print("Target distribution:")
print(proxy_df["target"].value_counts())

print("\nTarget rate:")
print(proxy_df["target"].mean())

Target distribution:
target
0    319373
1     12064
Name: count, dtype: int64

Target rate:
0.03639907433388547


In [9]:
# 4. Define X, y and groups

X = proxy_df[model_features].copy()

# Missing numeric values are filled with 0.
X = X.fillna(0)

y = proxy_df["target"]

groups = proxy_df["client_hash_id"]

print("X shape:", X.shape)

print("\nTarget distribution:")
print(y.value_counts())

X shape: (331437, 5)

Target distribution:
target
0    319373
1     12064
Name: count, dtype: int64


In [10]:
# 5. Client-grouped train/test split

from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print(
    "Training clients:",
    groups.iloc[train_idx].nunique()
)

print(
    "Test clients:",
    groups.iloc[test_idx].nunique()
)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTest target distribution:")
print(y_test.value_counts())

Training rows: 300880
Test rows: 30557
Training clients: 44
Test clients: 11

Training target distribution:
target
0    290333
1     10547
Name: count, dtype: int64

Test target distribution:
target
0    29040
1     1517
Name: count, dtype: int64


In [11]:
# 6. Train Random Forest classifier

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=25,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

print(rf)

RandomForestClassifier(class_weight='balanced', max_depth=10,
                       min_samples_leaf=25, n_estimators=200, n_jobs=-1,
                       random_state=42)


In [12]:
# 7. Generate ML scores

test_results = proxy_df.iloc[test_idx].copy()

test_results["ml_score"] = rf.predict_proba(
    X_test
)[:, 1]

print(test_results["ml_score"].describe())

count    30557.000000
mean         0.196106
std          0.309883
min          0.000000
25%          0.000002
50%          0.003328
75%          0.490494
max          0.998250
Name: ml_score, dtype: float64


In [13]:
# 8. Rank pages using ML score

test_results = test_results.sort_values(
    "ml_score",
    ascending=False
).reset_index(drop=True)

test_results["ml_rank"] = (
    np.arange(len(test_results)) + 1
)

ml_top20 = test_results.head(20)

print(
    ml_top20[
        [
            "ml_rank",
            "client_hash_id",
            "content_hash_id",
            "ml_score",
            "gsc_impressions",
            "gsc_ctr",
            "gsc_avg_position",
            "ga4_sessions",
            "engagement_rate"
        ]
    ]
)

    ml_rank           client_hash_id           content_hash_id  ml_score  \
0         1  client_e547b89c05043229  content_e986eb3ab70fcee6  0.998250   
1         2  client_e547b89c05043229  content_a49d621d476664e2  0.997071   
2         3  client_e547b89c05043229  content_805c0c2e555a8057  0.996840   
3         4  client_e547b89c05043229  content_9b76202a02885037  0.996757   
4         5  client_e547b89c05043229  content_e3b2a512ba3b320e  0.996756   
5         6  client_e547b89c05043229  content_19412009bb676d79  0.996685   
6         7  client_e547b89c05043229  content_10e8f76ed8c5c392  0.996550   
7         8  client_e547b89c05043229  content_448878c5eb8adc78  0.996472   
8         9  client_e547b89c05043229  content_cd33c3f6ef059886  0.996437   
9        10  client_e547b89c05043229  content_fecf82eefd6d232e  0.996239   
10       11  client_e547b89c05043229  content_80f1619d2f4038da  0.996107   
11       12  client_e547b89c05043229  content_23b3e0bbdd3b2cfb  0.996100   
12       13 

In [14]:
# 9. Compare ML ranking with baseline ranking

baseline_test = baseline_df.merge(
    test_results[
        [
            "client_hash_id",
            "content_hash_id"
        ]
    ],
    on=[
        "client_hash_id",
        "content_hash_id"
    ],
    how="inner"
)

baseline_test = baseline_test.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

baseline_test["baseline_rank"] = (
    np.arange(len(baseline_test)) + 1
)

comparison = test_results[
    [
        "client_hash_id",
        "content_hash_id",
        "ml_score",
        "ml_rank"
    ]
].merge(
    baseline_test[
        [
            "client_hash_id",
            "content_hash_id",
            "baseline_score",
            "baseline_rank"
        ]
    ],
    on=[
        "client_hash_id",
        "content_hash_id"
    ],
    how="inner"
)

print("Comparison rows:", len(comparison))

print(comparison.head(20))

Comparison rows: 30557
             client_hash_id           content_hash_id  ml_score  ml_rank  \
0   client_e547b89c05043229  content_e986eb3ab70fcee6  0.998250        1   
1   client_e547b89c05043229  content_a49d621d476664e2  0.997071        2   
2   client_e547b89c05043229  content_805c0c2e555a8057  0.996840        3   
3   client_e547b89c05043229  content_9b76202a02885037  0.996757        4   
4   client_e547b89c05043229  content_e3b2a512ba3b320e  0.996756        5   
5   client_e547b89c05043229  content_19412009bb676d79  0.996685        6   
6   client_e547b89c05043229  content_10e8f76ed8c5c392  0.996550        7   
7   client_e547b89c05043229  content_448878c5eb8adc78  0.996472        8   
8   client_e547b89c05043229  content_cd33c3f6ef059886  0.996437        9   
9   client_e547b89c05043229  content_fecf82eefd6d232e  0.996239       10   
10  client_e547b89c05043229  content_80f1619d2f4038da  0.996107       11   
11  client_e547b89c05043229  content_23b3e0bbdd3b2cfb  0.996100  

In [15]:
# 10. Spearman rank correlation

from scipy.stats import spearmanr

correlation, p_value = spearmanr(
    comparison["ml_rank"],
    comparison["baseline_rank"]
)

print("Spearman rank correlation:", correlation)
print("p-value:", p_value)

Spearman rank correlation: 0.25360582456051933
p-value: 0.0


In [16]:
# 11. Compare top-20 overlap

ml_top20_ids = set(
    comparison.nsmallest(20, "ml_rank")[
        "content_hash_id"
    ]
)

baseline_top20_ids = set(
    comparison.nsmallest(20, "baseline_rank")[
        "content_hash_id"
    ]
)

overlap = len(
    ml_top20_ids.intersection(baseline_top20_ids)
)

print("ML top-20:", len(ml_top20_ids))
print("Baseline top-20:", len(baseline_top20_ids))
print("Top-20 overlap:", overlap)

ML top-20: 20
Baseline top-20: 20
Top-20 overlap: 0


In [17]:
# 12. Feature importance
import pandas as pd
importance = pd.DataFrame({
    "feature": model_features,
    "importance": rf.feature_importances_
}).sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

print(importance)

            feature  importance
0  gsc_avg_position    0.680105
1   log_impressions    0.243511
2      log_sessions    0.047381
3           gsc_ctr    0.026355
4    log_engagement    0.002648


In [18]:
# 13. Save ML ranked queue

import os

os.makedirs(
    "work/outputs",
    exist_ok=True
)

output_path = (
    "work/outputs/ml_action_score.csv"
)

test_results.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Rows:", len(test_results))

Saved: work/outputs/ml_action_score.csv
Rows: 30557


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The model is most strongly influenced by `gsc_avg_position`, which accounts for approximately 68% of the feature importance. `log_impressions` is the second most important feature at approximately 24%. The remaining features — `log_sessions`, `gsc_ctr`, and `log_engagement` — have substantially smaller importance.

This suggests that the model primarily identifies pages that have reasonable search visibility but relatively poor search position. This is visible in the highest-ranked pages, which generally have thousands of impressions but low CTR and/or relatively poor average positions.

The model can therefore be wrong when a page has a poor average position but is not necessarily a high-priority content-refresh opportunity. For example, a page may rank poorly because it is targeting a highly competitive search query, rather than because its content is outdated or needs refreshing. Similarly, high impressions can cause a page to receive a high score even when there is limited evidence from engagement signals that the content itself needs attention.

Another limitation is that the model is learning the March 2026 proxy target rather than a future outcome such as whether a page actually improves after being refreshed. Therefore, a high model score should be interpreted as "similar to pages identified as review-worthy by the chosen proxy", rather than as a prediction that refreshing the page will definitely improve performance.

Overall, the model has learned a different and more flexible ranking from the hand-written baseline, with average search position and search visibility being its main signals. Its main weakness is that these signals do not necessarily distinguish between poor performance caused by content quality and poor performance caused by external factors such as search competition.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.